# Lab 01.1 — Modalità di esecuzione di Spark

## Introduzione
Un'applicazione Spark è composta da:

* **Driver**: il processo che crea la `SparkSession`, costruisce il piano di esecuzione (DAG) e distribuisce i task. Qui è il kernel di questo notebook.
* **Executor**: processi JVM che eseguono i task in parallelo e tengono in memoria i dati.
* **Cluster manager**: chi assegna le risorse agli executor.

Lo **stesso codice** può girare in modalità diverse cambiando solo il *master*:

| Master | Modalità | Dove girano gli executor |
|---|---|---|
| `local[*]` | Locale | thread dentro il driver (un solo container) |
| `spark://spark-master:7077` | Standalone | container `spark-worker` |
| `yarn` | Hadoop YARN | container del NodeManager (Lab 03) |

UI utili: Spark UI dell'applicazione http://localhost:4040 · Master Standalone http://localhost:8080

### Obiettivi
1. Importare le librerie e leggere il master configurato per il laboratorio.
2. Eseguire Spark in **modalità locale** (`local[*]`) e osservare dove girano i task.
3. Collegarsi al **cluster Standalone** e verificare come i task si distribuiscono sugli executor.
4. Capire la **lazy evaluation**: trasformazioni contro azioni, DAG e lineage.

## Step 1: Import e master configurato
Importiamo i moduli necessari e stampiamo la variabile d'ambiente `SPARK_MASTER`, impostata dal `docker-compose.yml` (per questo lab punta al cluster Standalone). Nei passi successivi indicheremo il master in modo esplicito, per confrontare le due modalità.

In [ ]:
import os, socket, time
from pyspark.sql import SparkSession

# Master di default del laboratorio, definito nel docker-compose.yml
print("Master configurato per questo laboratorio:", os.environ.get("SPARK_MASTER"))

## Step 2: Modalità locale `local[*]`
Driver ed executor sono nello **stesso processo JVM**; `*` = un thread per ogni core disponibile.
Ideale per sviluppo e test: nessun cluster necessario.
Creiamo la `SparkSession`, leggiamo il parallelismo di default e lanciamo 8 task che restituiscono il nome dell'host su cui girano: in locale sarà sempre quello del driver.

In [ ]:
# Sessione in modalità locale: nessun cluster, solo thread dentro il driver
spark = (SparkSession.builder
         .appName("Lab01-local")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext

print("Versione Spark      :", spark.version)
print("Master              :", sc.master)
print("Parallelismo default:", sc.defaultParallelism)

# Ogni task restituisce il nome dell'host su cui è stato eseguito
hosts = sc.parallelize(range(8), 8).map(lambda _: socket.gethostname()).collect()
print("Host che hanno eseguito i task:", set(hosts), "(sempre il driver)")

### Punti chiave (modalità locale)
* Con `local[*]` driver ed executor coincidono: tutti i task risultano eseguiti sullo stesso host.
* `defaultParallelism` è pari al numero di core visibili al driver.

## Step 3: Chiudere la sessione locale
Si può avere una sola `SparkContext` attiva per processo: prima di cambiare master dobbiamo fermare la sessione corrente con `spark.stop()`.

In [ ]:
# Libera la SparkContext: serve per ricreare la sessione con un altro master
spark.stop()

## Step 4: Modalità cluster Standalone `spark://spark-master:7077`
Il **master** riceve la richiesta di risorse e avvia un executor su ogni **worker**.
I task vengono eseguiti nei container `spark-worker` (si vedono host diversi).

> Disponibile in questo laboratorio (avviato con `docker compose up -d`).

Creiamo la sessione indicando le risorse di ciascun executor (768 MB e 1 core, compatibili con i worker da 1 GB): il parallelismo di default diventa il numero totale di core degli executor.

In [ ]:
# Sessione collegata al cluster Standalone, con le risorse richieste per executor
spark = (SparkSession.builder
         .appName("Lab01-standalone")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "768m")
         .config("spark.executor.cores", "1")
         .getOrCreate())
sc = spark.sparkContext

print("Master              :", sc.master)
print("Application ID      :", sc.applicationId)
print("Parallelismo default:", sc.defaultParallelism, "(= core totali degli executor)")

### Step 4b: Distribuzione dei task sugli executor
Ripetiamo l'esperimento dell'host: ora i task vengono eseguiti nei container `spark-worker`, quindi compaiono nomi di host diversi dal driver e il carico è ripartito tra gli executor.

In [ ]:
# Distribuzione dei task sugli executor
hosts = sc.parallelize(range(8), 8).map(lambda _: socket.gethostname()).collect()
for h in sorted(set(hosts)):
    print(f"executor su {h}: {hosts.count(h)} task")

### Step 4c: Executor registrati
Interroghiamo la JVM del driver (via `_jsc`) per sapere quante JVM sono attive. Il conteggio include il driver stesso, quindi con 2 worker ci aspettiamo 3. Lo stesso elenco è visibile nella scheda *Executors* della Spark UI.

In [ ]:
# Executor registrati (il driver compare anch'esso nell'elenco)
status = sc._jsc.sc().getExecutorMemoryStatus()
print("JVM attive (driver + executor):", status.size())

### Punti chiave (cluster Standalone)
* Cambiare il master è l'unica modifica necessaria: il codice dei task resta identico.
* In Standalone i task girano su host diversi dal driver, uno per ogni executor/worker.
* `defaultParallelism` riflette i core totali degli executor richiesti.

## Step 5: Lazy evaluation e DAG
Le **trasformazioni** (`map`, `filter`, ...) costruiscono solo un piano; il calcolo parte con un'**azione** (`count`, `collect`, `sum`, ...).
Misuriamo il tempo di definizione delle trasformazioni (quasi nullo) e quello dell'azione `count()` (che invia il job al cluster), poi stampiamo il **lineage** con `toDebugString()`.

In [ ]:
# Le trasformazioni costruiscono solo il piano: nessun calcolo, tempo ~0
t0 = time.time()
rdd = sc.parallelize(range(1, 5_000_001), 8).map(lambda x: x * x).filter(lambda x: x % 3 == 0)
print(f"Trasformazioni definite in {time.time() - t0:.4f} s  (nessun calcolo)")

# L'azione count() fa partire il job vero e proprio
t0 = time.time()
print("count =", rdd.count())
print(f"Azione eseguita in {time.time() - t0:.2f} s  (job inviato al cluster)")

# Lineage: la "ricetta" che Spark usa anche per ricalcolare le partizioni perse
print(rdd.toDebugString().decode())

### Punti chiave (lazy evaluation)
* Le trasformazioni sono istantanee perché registrano soltanto il piano (DAG).
* Solo un'azione avvia il calcolo distribuito.
* Il lineage permette a Spark di ricostruire le partizioni perse.

## Step 6: Chiudere la sessione
Terminiamo l'applicazione per liberare gli executor sul cluster: finché la sessione è aperta, i core restano riservati e i notebook successivi potrebbero non trovare risorse.

In [ ]:
# Rilascia le risorse del cluster
spark.stop()

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Master | `.master("local[*]")` / `.master("spark://spark-master:7077")` | sceglie dove girano gli executor |
| Sessione | `SparkSession.builder...getOrCreate()` | punto di ingresso dell'applicazione |
| Risorse executor | `spark.executor.memory`, `spark.executor.cores` | dimensiona gli executor sul cluster |
| Parallelismo | `sc.defaultParallelism` | numero di partizioni di default (core totali) |
| Distribuzione task | `sc.parallelize(...).map(...).collect()` | verifica su quali host girano i task |
| Lazy evaluation | trasformazioni vs azioni | il calcolo parte solo con un'azione |
| Lineage | `rdd.toDebugString()` | mostra la catena di trasformazioni (DAG) |
| Chiusura | `spark.stop()` | libera la SparkContext e le risorse del cluster |